In [1]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

In [2]:
from embedder import Embedder

embed = Embedder()

q1 = "How does approximate nearest neighbor search work?"

v1 = embed.encode(q1)

2026-06-29 14:14:58.563178205 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


In [3]:
v1[0]

np.float64(-0.02058203437252893)

In [4]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [5]:
len(documents)

72

In [6]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

In [7]:
filename_lst = [ item['filename'] for item in documents ]

In [8]:
filename_lst[0]

'01-agentic-rag/lessons/01-intro.md'

In [9]:
selected_filename = '02-vector-search/lessons/07-sqlitesearch-vector.md'
idx = filename_lst.index(selected_filename)

In [10]:
documents[idx]

{'content': '# Vector Search with sqlitesearch\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=csxKescwJYM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous section we used minsearch for vector search.\n\nIt works, but it has three problems:\n\n- It rebuilds the index on every startup\n- It keeps everything in memory\n- It searches by brute force\n\n\nWith text search we never felt these. Indexing was fast because we\ndidn\'t embed anything. With vector search, indexing runs a neural\nnetwork over every document, so it takes a minute on our dataset.\nKeeping everything in memory is fine here, but a larger dataset would\nneed too much space.\n\nThe third problem is brute-force search. For every query we compare the\nquery vector against every single document. With 1,000 documents this is\nfine, probably even faster than anything smarter. But as the dataset\ngrows past 10,000 or so, it gets slow, and we\'ll want an approximate\nmethod instead.\n\nWhat we\'ve done 

In [11]:
# embed content of selected page

pg = documents[idx]['content']

pg_v = embed.encode(pg)

In [16]:
# compute the cosine similarity with the query vector from Q1.

import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Reshape to 2D arrays (1 sample, N features)
vec1_2d = v1.reshape(1, -1)
vec2_2d = pg_v.reshape(1, -1)

# Compute similarity
similarity = cosine_similarity(vec1_2d, vec2_2d)
print(similarity[0][0])

0.36107027225589694


In [17]:
vec1_2d.shape

(1, 384)

In [18]:
v1

array([-2.05820344e-02, -1.40458849e-02,  3.02994061e-02, -5.40378445e-02,
        7.18781100e-02, -2.79537512e-02, -5.03093823e-02, -1.27217287e-02,
        4.08207902e-02, -2.60037446e-02,  3.05458646e-02,  4.21485309e-02,
        8.09861910e-02, -6.93957355e-02, -1.30190518e-01, -6.39247860e-02,
        4.81059741e-02,  1.60095554e-02, -5.22432468e-02, -7.13635281e-02,
       -3.83859209e-03,  2.48125508e-02,  4.40211692e-02, -3.45579077e-02,
        1.52686257e-02,  7.61533350e-03,  5.38177679e-02,  1.18252557e-02,
        1.32434005e-02,  2.89461963e-02,  6.64912054e-03,  7.04788357e-02,
        6.19290508e-02,  2.11051780e-02, -7.33482889e-02,  2.84129213e-02,
       -3.72108635e-02,  6.22799314e-02, -4.86973136e-02,  4.49663910e-02,
       -2.59481060e-02,  2.04324269e-02,  1.79650458e-02,  1.02705602e-02,
        2.74898095e-03,  2.84324992e-02, -3.30318167e-02,  6.70969402e-02,
       -1.56520555e-02, -8.51907638e-02, -1.24307135e-01,  4.32504103e-02,
       -5.64433014e-02,  

In [19]:
vec1_2d

array([[-2.05820344e-02, -1.40458849e-02,  3.02994061e-02,
        -5.40378445e-02,  7.18781100e-02, -2.79537512e-02,
        -5.03093823e-02, -1.27217287e-02,  4.08207902e-02,
        -2.60037446e-02,  3.05458646e-02,  4.21485309e-02,
         8.09861910e-02, -6.93957355e-02, -1.30190518e-01,
        -6.39247860e-02,  4.81059741e-02,  1.60095554e-02,
        -5.22432468e-02, -7.13635281e-02, -3.83859209e-03,
         2.48125508e-02,  4.40211692e-02, -3.45579077e-02,
         1.52686257e-02,  7.61533350e-03,  5.38177679e-02,
         1.18252557e-02,  1.32434005e-02,  2.89461963e-02,
         6.64912054e-03,  7.04788357e-02,  6.19290508e-02,
         2.11051780e-02, -7.33482889e-02,  2.84129213e-02,
        -3.72108635e-02,  6.22799314e-02, -4.86973136e-02,
         4.49663910e-02, -2.59481060e-02,  2.04324269e-02,
         1.79650458e-02,  1.02705602e-02,  2.74898095e-03,
         2.84324992e-02, -3.30318167e-02,  6.70969402e-02,
        -1.56520555e-02, -8.51907638e-02, -1.24307135e-0

In [20]:
# Q3
# We chunk the pages.

from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

In [22]:
chunks[0]['content']

'# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simple language 

In [23]:
# We embed every chunk's content with encode_batch, 
# stack the vectors into a matrix X, 
# and score the Q1 query against all chunks

from tqdm.auto import tqdm
import numpy as np

texts = [ chunk['content'] for chunk in chunks ]

batch_size = 50
X = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    X.extend(batch_vectors)

X = np.array(X)

  0%|          | 0/6 [00:00<?, ?it/s]

In [24]:
query = "How does approximate nearest neighbor search work?"
v_query = embed.encode(query)

scores = X.dot(v_query)
idx = np.argmax(scores)

chunks[idx]['filename']

'02-vector-search/lessons/07-sqlitesearch-vector.md'

In [26]:
# Q4: Vector search with minsearch

from minsearch import VectorSearch

vindex = VectorSearch()
vindex.fit(X, chunks)

In [27]:
q1 = "What metric do we use to evaluate a search engine?"
v1 = v1 = embed.encode(q1)

vindex.search(v1, num_results=5)

[{'start': 0,
  'content': "# Search Evaluation Metrics\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=TuirMy3Pdbk&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we computed relevance lists for search results.\nWe can turn those lists into metrics.\n\n## Hit Rate\n\nHit Rate (also called Recall@k) measures the fraction of queries where\nthe correct document appears anywhere in the results:\n\n```python\nexample = [\n    [1, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 1, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n]\n```\n\nEach line is one query. If a line contains `1`, search found the\ncorrect document somewhere in the top 5 results. If the line contains\nonly zeros, search did not find the correct document.\n\nIn our set

In [28]:
# Q5. Text search vs vector search
# Index the same chunks with Index from minsearch. Use content as a text field.

from minsearch import Index

index = Index(
    text_fields=["content"]
)

index.fit(chunks)

In [29]:
query = "How do I store vectors in PostgreSQL?"

In [31]:
# Run text search for this query

search_results = index.search(
    query,
    num_results=5
)

search_results

[{'start': 4000,
  'content': 'get 0.01.\n\nThe first score for `q1` vs `d` (0.32) is higher, so that query is more\nsimilar to the document about registration. The second score for `q2`\nvs `d` sits near 0, because installing Docker has nothing to do with\nregistration. A score near 0 means the two vectors are about as\ndifferent as they can be.\n\nThat\'s the whole idea behind vector search: similar texts get similar\nvectors, and a dot product tells us how similar.\n\n## Cosine similarity\n\nThe `all-MiniLM-L6-v2` model outputs normalized vectors - vectors with\nunit length. When both vectors are normalized, the dot product equals\ncosine similarity. That\'s why the model documentation says it "uses\ncosine similarity."\n\nCosine similarity measures the angle between two vectors, ignoring\ntheir length:\n\n- 1.0 = same direction (similar)\n- 0.0 = perpendicular (unrelated)\n- -1.0 = opposite direction (opposite meaning)\n\nFormally, if `theta` is the angle between two vectors, cosin

In [33]:
# Run vector search for this query

query_v = embed.encode(query)

vec_search = vindex.search(query_v, num_results=5)

vec_search

[{'start': 0,
  'content': '# Vector Search with PGVector\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=0P54MFyz-mc&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nMany real databases can do vector search. Elasticsearch has it, and\nthere are dedicated stores like Qdrant and Chroma. We\'ll go with\nPostgres. Most of us already run it at work, and the data engineering\ncourse uses it too. The concept is the same as with sqlitesearch, only\nthe database under the hood changes.\n\n[pgvector](https://github.com/pgvector/pgvector) is the PostgreSQL\nextension that makes this work. Install it and Postgres can do vector\nsimilarity search. On top of that you get the usual production features,\nlike concurrent access, transactions, and large datasets.\n\nWe\'ll run Postgres with pgvector in Docker.\n\n## Starting Postgres with pgvector\n\nPull the image and start a container:\n\n```bash\ndocker run -it \\\n    --name pgvector \\\n    -e POSTGRES_USER=user \\\n    -e POSTGRES_PASSWO

In [34]:
vector_search_filenames = [ result['filename'] for result in vec_search ]
vector_search_filenames

['02-vector-search/lessons/08-pgvector.md',
 '02-vector-search/lessons/08-pgvector.md',
 '03-orchestration/lessons/05-rag.md',
 '02-vector-search/lessons/08-pgvector.md',
 '02-vector-search/lessons/08-pgvector.md']

In [35]:
search_results_filenames = [ result['filename'] for result in search_results ]
search_results_filenames

['02-vector-search/lessons/02-embeddings.md',
 '03-orchestration/lessons/05-rag.md',
 '02-vector-search/lessons/01-intro.md',
 '03-orchestration/lessons/05-rag.md',
 '02-vector-search/lessons/01-intro.md']

In [36]:
# Which file shows up in the vector results but not in the text results?
# 02-vector-search/lessons/08-pgvector.md

set(vector_search_filenames) - set(search_results_filenames)

{'02-vector-search/lessons/08-pgvector.md'}

In [37]:
# Q6: Hybrid Search
# Firstly, define reciprocal rank fusion function

def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [38]:
# This is the next query.

query2 = "How do I give the model access to tools?"

In [41]:
# Run text search on this query

text_results = index.search(
    query2,
    num_results=len(chunks)
    
)

len(text_results)

295

In [42]:
# Run vector search on query2

query2_v = embed.encode(query2)

vector_results = vindex.search(query2_v, num_results=len(chunks))

len(vector_results)

237

In [43]:
results = rrf([vector_results, text_results])

In [44]:
results[0]

{'start': 4000,
 'content': ' function. `parameters` is a JSON schema\nfor the arguments, and we mark `query` as required so the model always\nfills it in.\n\n## Sending the question with the tool\n\nNow we send the same question as before, but this time we include the\ntool in the request:\n\n```python\nresponse = openai_client.responses.create(\n    model="gpt-5.4-mini",\n    input=messages,\n    tools=[search_tool],\n)\n\nresponse.output\n```\n\nLook at the output. Instead of a message with the answer, the response\ncontains a `function_call` entry. The model decided it needs to search\nthe FAQ before answering. Rather than reply, it asked us to run the\nsearch function first.\n\nLook at the arguments too. The model didn\'t pass our question\nverbatim. It judged the raw question wasn\'t the best query to search\nwith. So it rewrote our enrollment question into search keywords like\n"enroll late join course".\n\n## Executing the function and sending the result back\n\nThe function ca

In [45]:
vector_results[:2]

[{'start': 2000,
  'content': 'wrong.\n\n## The project\n\nRAG solves these problems by giving the LLM relevant documents at\nquestion time. We don\'t hope the model memorized the answer. We\nretrieve the right information and hand it to the LLM, and the model\ngenerates a grounded response. This lets us inject knowledge the model\nnever saw during training. That\'s why RAG is still the most common way\npeople use LLMs in the industry.\n\nTo make this concrete, we build a FAQ agent for our course. A student\nasks something like "when does the course start?" and the agent answers\nfrom the FAQ data we prepared.\n\nThis module has two parts.\n\nIn Part 1 (the next 9 lessons) we will:\n\n- Understand what RAG is and how it works\n- Build a search engine over a real FAQ dataset\n- Write a prompt that combines the user\'s question with search results\n- Wire it all together into a working RAG pipeline\n- Split ingestion and query into separate processes\n\nIn Part 2, we make the pipeline ag

In [46]:
text_results[:2]

[{'start': 0,
  'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can